In [5]:
# ============================================================
# Cell 0 — Install dependencies
# Run this cell first. Takes ~3 minutes on Kaggle.
# ============================================================
!pip install -q \
    datasets>=2.18.0 \
    huggingface_hub>=0.22.0 \
    tqdm

In [31]:
# ============================================================
# Cell 1 — Paths and directory setup
# ============================================================
import os, sys

INPUT_BASE   = "/kaggle/input/datasets/giahuyvotran"
OUTPUT_BASE  = "/kaggle/working"
RAW_DIR      = f"{OUTPUT_BASE}/data_raw"
LOG_DIR      = f"{OUTPUT_BASE}/logs"
HF_CACHE_DIR = f"{OUTPUT_BASE}/hf_cache"
ZALO_OUT     = f"{RAW_DIR}/zalo"
QUANGBUT_OUT = f"{RAW_DIR}/quangbut"
THINH_OUT    = f"{RAW_DIR}/thinh4526"
TUONG_OUT    = f"{RAW_DIR}/tuongdang"
HF_OUT       = f"{RAW_DIR}/huggingface"

for d in [RAW_DIR, LOG_DIR, HF_CACHE_DIR,
          ZALO_OUT, QUANGBUT_OUT, THINH_OUT, TUONG_OUT, HF_OUT]:
    os.makedirs(d, exist_ok=True)

print("Python:", sys.version)
print("Output base:", OUTPUT_BASE)
print("Directories created successfully.")

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Output base: /kaggle/working
Directories created successfully.


In [32]:
# ============================================================
# Cell 2 — Inspect Kaggle input datasets
# ============================================================
import os

def list_kaggle_inputs(base_path="/kaggle/input"):
    result = {}
    if not os.path.exists(base_path):
        print(f"[WARN] {base_path} does not exist.")
        return result
    for name in os.listdir(base_path):
        p = os.path.join(base_path, name)
        files = []
        for root, _, fnames in os.walk(p):
            for f in fnames:
                fp = os.path.join(root, f)
                files.append((fp, round(os.path.getsize(fp) / 1024**2, 2)))
        result[name] = files
    return result

kaggle_inputs = list_kaggle_inputs()
total_mb = 0
for ds, files in kaggle_inputs.items():
    print(f"\n--- {ds} ---")
    for path, mb in sorted(files, key=lambda x: -x[1]):
        print(f"  {path:<80} {mb:>8.2f} MB")
        total_mb += mb
print(f"\nTotal input size: {total_mb:.2f} MB")


--- datasets ---
  /kaggle/input/datasets/quangbut/vietnamese-legal/sent_truncated_vbpl_update.csv   1826.76 MB
  /kaggle/input/datasets/quangbut/vietnamese-legal/legal_truncated_corpus.csv       1692.94 MB
  /kaggle/input/datasets/quangbut/vietnamese-legal/vbpl_crawl.json                  1618.12 MB
  /kaggle/input/datasets/tuongdang/vietnamese-legal-dataset/vbpl_crawl_2.csv        1533.87 MB
  /kaggle/input/datasets/quangbut/vietnamese-legal/vbpl.csv                         1486.53 MB
  /kaggle/input/datasets/quangbut/vietnamese-legal/bm25_legal_corpus                 855.17 MB
  /kaggle/input/datasets/lookingformyself/zac2021-ltr-data/bm_25_pairs_top30         308.79 MB
  /kaggle/input/datasets/hmthanh/legal-text-retrieval/pair_data/bm_25_pairs_top20    257.51 MB
  /kaggle/input/datasets/hariwh0/zaloai2021-legal-text-retrieval/legal_corpus_splitted.csv   206.68 MB
  /kaggle/input/datasets/lookingformyself/zac2021-ltr-data/legal_dict.json           169.01 MB
  /kaggle/input/datasets

In [33]:
# ============================================================
# Cell 3 — Locate and register Kaggle dataset files
# ============================================================
import os, glob, json

KAGGLE_REGISTRY = {}

zalo_candidates = [
    "/kaggle/input/datasets/hariwh0/zaloai2021-legal-text-retrieval",
    "/kaggle/input/datasets/lookingformyself/zac2021-ltr-data",
]
for c in zalo_candidates:
    if os.path.exists(c):
        KAGGLE_REGISTRY["zalo"] = c
        print(f"[OK] Zalo AI 2021: {c}")
        break
else:
    print("[MISSING] Zalo AI 2021")

quangbut_path = "/kaggle/input/datasets/quangbut/vietnamese-legal"
if os.path.exists(quangbut_path):
    KAGGLE_REGISTRY["quangbut"] = quangbut_path
    print(f"[OK] quangbut: {quangbut_path}")
else:
    print("[MISSING] quangbut/vietnamese-legal")

tuong_path = "/kaggle/input/datasets/tuongdang/vietnamese-legal-dataset"
if os.path.exists(tuong_path):
    KAGGLE_REGISTRY["tuongdang"] = tuong_path
    print(f"[OK] tuongdang: {tuong_path}")
else:
    print("[MISSING] tuongdang/vietnamese-legal-dataset")

print("\nRegistry:", json.dumps(KAGGLE_REGISTRY, indent=2))

[OK] Zalo AI 2021: /kaggle/input/datasets/hariwh0/zaloai2021-legal-text-retrieval
[OK] quangbut: /kaggle/input/datasets/quangbut/vietnamese-legal
[OK] tuongdang: /kaggle/input/datasets/tuongdang/vietnamese-legal-dataset

Registry: {
  "zalo": "/kaggle/input/datasets/hariwh0/zaloai2021-legal-text-retrieval",
  "quangbut": "/kaggle/input/datasets/quangbut/vietnamese-legal",
  "tuongdang": "/kaggle/input/datasets/tuongdang/vietnamese-legal-dataset"
}


In [34]:
# ============================================================
# Cell 4 — Copy Kaggle dataset files to RAW_DIR
# ============================================================
import shutil, glob, os

def copy_dataset_files(src_dir, dst_dir, extensions=[".json", ".jsonl", ".csv", ".txt"]):
    copied = []
    if not os.path.exists(src_dir):
        print(f"  [SKIP] {src_dir}")
        return copied
    for ext in extensions:
        for src_path in glob.glob(f"{src_dir}/**/*{ext}", recursive=True):
            rel_path  = os.path.relpath(src_path, src_dir)
            dst_path  = os.path.join(dst_dir, rel_path)
            os.makedirs(os.path.dirname(dst_path), exist_ok=True)
            shutil.copy2(src_path, dst_path)
            mb = os.path.getsize(dst_path) / 1024**2
            print(f"  Copied: {rel_path:<60} ({mb:.2f} MB)")
            copied.append(dst_path)
    return copied

copy_log = {}
if "zalo"     in KAGGLE_REGISTRY: copy_log["zalo"]     = copy_dataset_files(KAGGLE_REGISTRY["zalo"],     ZALO_OUT)
if "quangbut" in KAGGLE_REGISTRY: copy_log["quangbut"] = copy_dataset_files(KAGGLE_REGISTRY["quangbut"], QUANGBUT_OUT)
if "tuongdang"in KAGGLE_REGISTRY: copy_log["tuongdang"]= copy_dataset_files(KAGGLE_REGISTRY["tuongdang"],TUONG_OUT)
print(f"\nTotal files copied: {sum(len(v) for v in copy_log.values())}")

  Copied: public_test_sample_submission.json                           (0.05 MB)
  Copied: train_question_answer.json                                   (1.23 MB)
  Copied: legal_corpus.json                                            (113.90 MB)
  Copied: public_test_question.json                                    (0.11 MB)
  Copied: train_qna.csv                                                (0.70 MB)
  Copied: legal_corpus_hashmap.csv                                     (16.20 MB)
  Copied: legal_corpus_legend.csv                                      (0.15 MB)
  Copied: legal_corpus_merged_u256.csv                                 (148.75 MB)
  Copied: legal_corpus_splitted.csv                                    (206.68 MB)
  Copied: legal_corpus_original.csv                                    (136.19 MB)
  Copied: legal_corpus_merged_u369.csv                                 (148.54 MB)
  Copied: stopwords.txt                                                (0.02 MB)
  Copied: dvc_tes

In [35]:
# ============================================================
# Cell 4b — Load HuggingFace token from Kaggle Secrets
# Prerequisite: add HF_TOKEN in Kaggle Notebook -> Add-ons -> Secrets
# ============================================================
from kaggle_secrets import UserSecretsClient
user_secrets  = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
print("HF_TOKEN loaded:", secret_value_0[:8] + "..." if secret_value_0 else "[EMPTY — check Secrets]")

HF_TOKEN loaded: hf_vnPbq...


In [36]:
# ============================================================
# Cell 5 — Download HuggingFace datasets
# FIX 1: HF_TOKEN now correctly assigned from secret_value_0
# FIX 2: Removed trust_remote_code (deprecated in new HF versions)
# FIX 3: Removed test split for vietnamese-legal-qa (only train exists)
# ============================================================
import os, json
from datasets import load_dataset

# FIX 1: correct token assignment
os.environ["HF_TOKEN"] = secret_value_0


def save_hf_dataset_as_jsonl(dataset_name, config, split, out_dir, filename):
    print(f"  Loading {dataset_name} / split={split} ...")
    try:
        # FIX 2: removed trust_remote_code=True
        ds = load_dataset(dataset_name, config, split=split, cache_dir=HF_CACHE_DIR) \
             if config else \
             load_dataset(dataset_name, split=split, cache_dir=HF_CACHE_DIR)
    except Exception as e:
        print(f"  [ERROR] {e}")
        return 0
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, filename)
    count = 0
    with open(out_path, "w", encoding="utf-8") as f:
        for record in ds:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            count += 1
    print(f"  [OK] {count:,} records -> {out_path} ({os.path.getsize(out_path)/1024**2:.2f} MB)")
    return count


hf_download_log = {}

# 1. Vietnamese Legal QA — FIX 3: only train split (test does not exist)
print("\n[thangvip/vietnamese-legal-qa]")
out_dir = f"{HF_OUT}/vietnamese-legal-qa"
n = save_hf_dataset_as_jsonl("thangvip/vietnamese-legal-qa", None, "train", out_dir, "train.jsonl")
hf_download_log["vietnamese-legal-qa"] = n

# 2. Vietnamese Legal Chat
print("\n[luanngo/Vietnamese-Legal-Chat-Dataset]")
out_dir = f"{HF_OUT}/vietnamese-legal-chat"
n = save_hf_dataset_as_jsonl("luanngo/Vietnamese-Legal-Chat-Dataset", None, "train", out_dir, "train.jsonl")
hf_download_log["vietnamese-legal-chat"] = n

# 3. Large VN Legal Queries — streaming to avoid OOM
print("\n[phamson02/large-vi-legal-queries]")
try:
    out_dir  = f"{HF_OUT}/large-vi-legal-queries"
    os.makedirs(out_dir, exist_ok=True)
    out_path = f"{out_dir}/train.jsonl"
    ds_stream = load_dataset("phamson02/large-vi-legal-queries",
                             split="train", streaming=True, cache_dir=HF_CACHE_DIR)
    MAX_ROWS = 500_000
    count = 0
    with open(out_path, "w", encoding="utf-8") as f:
        for record in ds_stream:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            count += 1
            if count >= MAX_ROWS:
                break
    print(f"  [OK] {count:,} records -> {out_path} ({os.path.getsize(out_path)/1024**2:.2f} MB)")
    hf_download_log["large-vi-legal-queries"] = count
except Exception as e:
    print(f"  [ERROR] {e}")
    hf_download_log["large-vi-legal-queries"] = 0

# 4. ViBidLQA_v1 — all 3 splits
print("\n[ntphuc149/ViBidLQA_v1]")
out_dir = f"{HF_OUT}/ViBidLQA_v1"
hf_download_log["ViBidLQA_v1"] = 0
for split in ["train", "test", "validation"]:
    try:
        n_split = save_hf_dataset_as_jsonl("ntphuc149/ViBidLQA_v1", None, split, out_dir, f"{split}.jsonl")
        hf_download_log["ViBidLQA_v1"] += n_split
    except Exception as e:
        print(f"  [SKIP] {split}: {e}")

print("\nHuggingFace download summary:")
for name, count in hf_download_log.items():
    print(f"  {name:<35}: {count:>8,} records")


[thangvip/vietnamese-legal-qa]
  Loading thangvip/vietnamese-legal-qa / split=train ...
  [OK] 9,715 records -> /kaggle/working/data_raw/huggingface/vietnamese-legal-qa/train.jsonl (67.78 MB)

[luanngo/Vietnamese-Legal-Chat-Dataset]
  Loading luanngo/Vietnamese-Legal-Chat-Dataset / split=train ...


Repo card metadata block was not found. Setting CardData to empty.


  [OK] 3,537 records -> /kaggle/working/data_raw/huggingface/vietnamese-legal-chat/train.jsonl (8.98 MB)

[phamson02/large-vi-legal-queries]
  [OK] 500,000 records -> /kaggle/working/data_raw/huggingface/large-vi-legal-queries/train.jsonl (904.62 MB)

[ntphuc149/ViBidLQA_v1]
  Loading ntphuc149/ViBidLQA_v1 / split=train ...
  [OK] 1,928 records -> /kaggle/working/data_raw/huggingface/ViBidLQA_v1/train.jsonl (2.64 MB)
  Loading ntphuc149/ViBidLQA_v1 / split=test ...
  [OK] 603 records -> /kaggle/working/data_raw/huggingface/ViBidLQA_v1/test.jsonl (0.79 MB)
  Loading ntphuc149/ViBidLQA_v1 / split=validation ...
  [OK] 482 records -> /kaggle/working/data_raw/huggingface/ViBidLQA_v1/validation.jsonl (0.64 MB)

HuggingFace download summary:
  vietnamese-legal-qa                :    9,715 records
  vietnamese-legal-chat              :    3,537 records
  large-vi-legal-queries             :  500,000 records
  ViBidLQA_v1                        :    3,013 records


In [37]:
# ============================================================
# Cell 6 — Zalo AI 2021: Inspect legal_corpus
# FIX: Flattens documents -> articles (Điều).
# legal_corpus.json has 3,271 documents, NOT 3,271 articles.
# Each document contains a list of articles under doc["articles"].
# The true article count is ~224,000+.
# ============================================================
import json, os, glob

def find_zalo_corpus(raw_zalo_dir):
    hits = glob.glob(f"{raw_zalo_dir}/**/legal_corpus.json", recursive=True)
    return hits[0] if hits else None

corpus_path = find_zalo_corpus(ZALO_OUT)
all_articles = []   # exposed to Cell 8

if corpus_path is None:
    print("[WARN] legal_corpus.json not found.")
    for f in glob.glob(f"{ZALO_OUT}/**/*", recursive=True):
        if os.path.isfile(f): print(f"  {f}")
else:
    print(f"[OK] {corpus_path}  ({os.path.getsize(corpus_path)/1024**2:.2f} MB)")

    with open(corpus_path, "r", encoding="utf-8") as f:
        try:
            corpus_data = json.load(f)
        except json.JSONDecodeError:
            f.seek(0)
            corpus_data = [json.loads(l) for l in f if l.strip()]

    print(f"\n  Top-level documents (van ban): {len(corpus_data):,}")
    print(f"  Sample doc keys: {list(corpus_data[0].keys())}")

    # ---- FIX: Flatten doc["articles"] to get true article count ----
    for doc in corpus_data:
        law_id = doc.get("law_id", "unknown")
        for article in doc.get("articles", []):
            all_articles.append({
                "law_id":     law_id,
                "article_id": article.get("article_id", ""),
                "title":      article.get("title", ""),
                "text":       article.get("text", "")
            })

    print(f"  Total articles (Dieu) after flatten: {len(all_articles):,}")

    # Sample articles
    print("\n  Sample articles:")
    for a in all_articles[:3]:
        print(f"    [{a['law_id']}] {a['title'][:70]}")
        print(f"    text[:80]: {a['text'][:80]}")

    # Article-level length stats
    art_lengths = [len(a["text"]) for a in all_articles]
    art_tokens  = [len(a["text"].split()) for a in all_articles]
    print(f"\n  Article text length (chars):")
    print(f"    Min:    {min(art_lengths):,}")
    print(f"    Max:    {max(art_lengths):,}")
    print(f"    Mean:   {sum(art_lengths)//len(art_lengths):,}")
    print(f"    Median: {sorted(art_lengths)[len(art_lengths)//2]:,}")
    print(f"    Total chars:  {sum(art_lengths):,}")
    print(f"    Est. words:   {sum(art_tokens):,}")

[OK] /kaggle/working/data_raw/zalo/legal_corpus.json  (113.90 MB)

  Top-level documents (van ban): 3,271
  Sample doc keys: ['law_id', 'articles']
  Total articles (Dieu) after flatten: 61,425

  Sample articles:
    [01/2009/tt-bnn] Điều 1. Phạm vi áp dụng
    text[:80]: Thông tư này hướng dẫn tuần tra, canh gác bảo vệ đê Điều trong mùa lũ đối với cá
    [01/2009/tt-bnn] Điều 2. Tổ chức lực lượng
    text[:80]: 1. Hàng năm trước mùa mưa, lũ, Ủy ban nhân dân cấp xã nơi có đê phải tổ chức lực
    [01/2009/tt-bnn] Điều 3. Tiêu chuẩn của các thành viên thuộc lực lượng tuần tra, canh g
    text[:80]: 1. Là người khoẻ mạnh, tháo vát, đủ khả năng đảm đương những công việc nặng nhọc

  Article text length (chars):
    Min:    0
    Max:    252,881
    Mean:   1,298
    Median: 829
    Total chars:  79,756,393
    Est. words:   17,682,365


In [38]:
# ============================================================
# Cell 7 — Inspect HuggingFace dataset schemas
# ============================================================
import json, os

def inspect_jsonl(path, n_samples=1):
    if not os.path.exists(path):
        return {"status": "NOT FOUND", "path": path}
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= n_samples: break
            try: records.append(json.loads(line))
            except: continue
    total = sum(1 for _ in open(path, encoding="utf-8"))
    return {
        "status": "OK", "path": path, "total": total,
        "columns": list(records[0].keys()) if records else [],
        "samples": records
    }

targets = [
    ("VN Legal QA",       f"{HF_OUT}/vietnamese-legal-qa/train.jsonl"),
    ("VN Legal Chat",     f"{HF_OUT}/vietnamese-legal-chat/train.jsonl"),
    ("Large VN Queries",  f"{HF_OUT}/large-vi-legal-queries/train.jsonl"),
    ("ViBidLQA_v1 train", f"{HF_OUT}/ViBidLQA_v1/train.jsonl"),
    ("ViBidLQA_v1 val",   f"{HF_OUT}/ViBidLQA_v1/validation.jsonl"),
    ("ViBidLQA_v1 test",  f"{HF_OUT}/ViBidLQA_v1/test.jsonl"),
]
for label, path in targets:
    info = inspect_jsonl(path)
    print(f"\n{'='*55}")
    print(f"Dataset : {label}")
    print(f"Status  : {info['status']}")
    if info["status"] == "OK":
        print(f"Records : {info['total']:,}")
        print(f"Columns : {info['columns']}")
        s = json.dumps(info['samples'][0], ensure_ascii=False)
        print(f"Sample  : {s[:300]}{'...' if len(s)>300 else ''}")


Dataset : VN Legal QA
Status  : OK
Records : 9,715
Columns : ['doc_name', 'doc_type_name', 'article_content', 'generated_qa_pairs', 'generation_time']
Sample  : {"doc_name": "Luật Địa chất và Khoáng sản của Quốc hội, số 54/2024/QH15", "doc_type_name": "Luật", "article_content": "Điều 5. Nguyên tắc hội nhập và hợp tác quốc tế về địa chất, khoáng sản\n\n1. Hội nhập và hợp tác quốc tế trong nghiên cứu, điều tra cơ bản địa chất, điều tra địa chất về khoáng sản,...

Dataset : VN Legal Chat
Status  : OK
Records : 3,537
Columns : ['conversations']
Sample  : {"conversations": [{"from": "human", "value": "Dưới đây là các câu hỏi trắc nghiệm (có đáp án) về legal multiple choice. Vui lòng chọn đáp án phù hợp nhất cho câu hỏi này.\n\nCâu hỏi: Tại sao khi đi bầu cử, người ta lại yêu cầu chúng ta viết phiếu riêng tư và không cho ai xem?\nĐể đảm bảo quyền tự d...

Dataset : Large VN Queries
Status  : OK
Records : 500,000
Columns : ['passage_id', 'domain', 'title', 'header', 'context', 'aspect', 'que

In [39]:
# ============================================================
# Cell 8 — Token count estimation (corrected)
# FIX 1: vietnamese-legal-qa uses correct fields:
#         article_content (str) + generated_qa_pairs (list of dicts)
# FIX 2: large-vi-legal-queries uses correct fields: context, query, title, header
# FIX 3: Zalo token count uses flattened all_articles (not document-level)
# FIX 4: Added quangbut CSV/JSON token estimation via sampling
# ============================================================
import os, json, glob
import pandas as pd


def estimate_tokens_jsonl(path, text_fields):
    """Whitespace-split token estimate. Handles str, list-of-str, list-of-dict."""
    if not os.path.exists(path):
        return 0
    total = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try: record = json.loads(line)
            except: continue
            for field in text_fields:
                val = record.get(field, "")
                if isinstance(val, str):
                    total += len(val.split())
                elif isinstance(val, list):
                    for item in val:
                        if isinstance(item, dict):
                            # list of dicts: extract all string values
                            for v in item.values():
                                if isinstance(v, str):
                                    total += len(v.split())
                        else:
                            total += len(str(item).split())
    return total


def estimate_tokens_csv_sampled(path, sample_rows=2000):
    """Sample-based token estimate for large CSV files. Extrapolates from file size."""
    if not os.path.exists(path):
        return 0
    file_size_bytes = os.path.getsize(path)
    try:
        sample = pd.read_csv(path, nrows=sample_rows, on_bad_lines="skip", low_memory=False)
        text_sample = " ".join(
            sample.select_dtypes(include="object").fillna("").values.flatten().astype(str)
        )
        sample_tokens = len(text_sample.split())
        sample_bytes  = len(text_sample.encode("utf-8"))
        if sample_bytes == 0:
            return 0
        return int(sample_tokens * (file_size_bytes / sample_bytes))
    except Exception as e:
        print(f"  [WARN] CSV sampling failed for {os.path.basename(path)}: {e}")
        return 0


token_stats = {}

# --- FIX 3: Zalo — use flattened all_articles (not document-level estimate) ---
if all_articles:
    zalo_tokens = sum(len(a["text"].split()) for a in all_articles)
    token_stats["zalo_articles"] = zalo_tokens
    print(f"Zalo articles: {len(all_articles):,} articles, {zalo_tokens:,} words")

# --- HuggingFace datasets with FIXED field names ---
hf_token_targets = [
    # FIX 1: use article_content + generated_qa_pairs (list of dicts)
    ("vietnamese-legal-qa",    f"{HF_OUT}/vietnamese-legal-qa/train.jsonl",
     ["article_content", "generated_qa_pairs"]),
    # conversations is list of dicts with 'value' key
    ("vietnamese-legal-chat",  f"{HF_OUT}/vietnamese-legal-chat/train.jsonl",
     ["conversations"]),
    # FIX 2: correct fields from schema inspection
    ("large-vi-legal-queries", f"{HF_OUT}/large-vi-legal-queries/train.jsonl",
     ["context", "query", "title", "header"]),
    ("ViBidLQA_v1_train",      f"{HF_OUT}/ViBidLQA_v1/train.jsonl",
     ["question", "answer", "context"]),
]
for name, path, fields in hf_token_targets:
    t = estimate_tokens_jsonl(path, fields)
    token_stats[name] = t
    print(f"{name:<35}: {t:,} words")

# --- FIX 4: quangbut CSV/JSON — sample-based estimation ---
print("\nSampling quangbut files (takes ~1 min)...")
for csv_file in glob.glob(f"{QUANGBUT_OUT}/**/*.csv", recursive=True):
    fname = os.path.basename(csv_file)
    mb = os.path.getsize(csv_file) / 1024**2
    t = estimate_tokens_csv_sampled(csv_file)
    token_stats[f"quangbut_{fname}"] = t
    print(f"  {fname:<50} ({mb:.0f} MB) -> est. {t:,} words")

# Also count tuongdang CSV
for csv_file in glob.glob(f"{TUONG_OUT}/**/*.csv", recursive=True):
    fname = os.path.basename(csv_file)
    mb = os.path.getsize(csv_file) / 1024**2
    t = estimate_tokens_csv_sampled(csv_file)
    token_stats[f"tuongdang_{fname}"] = t
    print(f"  {fname:<50} ({mb:.0f} MB) -> est. {t:,} words")

# --- Summary ---
print("\nToken count estimates (whitespace split, multiply ~1.3 for BPE):")
print(f"{'Dataset':<40} {'Words':>14} {'Est. BPE':>14}")
print("-" * 72)
grand_total = 0
for name, tokens in sorted(token_stats.items(), key=lambda x: -x[1]):
    bpe = int(tokens * 1.3)
    grand_total += bpe
    print(f"{name:<40} {tokens:>14,} {bpe:>14,}")
print("-" * 72)
print(f"{'TOTAL':<40} {'':>14} {grand_total:>14,}")
print(f"\nEst. total BPE tokens : {grand_total:,}")
print(f"CPT target            : 200,000,000")
if grand_total >= 200_000_000:
    print("[PASS] Sufficient data for CPT.")
else:
    print(f"[GAP]  Need ~{200_000_000 - grand_total:,} more tokens.")

Zalo articles: 61,425 articles, 17,682,365 words
vietnamese-legal-qa                : 10,965,772 words
vietnamese-legal-chat              : 1,518,048 words
large-vi-legal-queries             : 137,030,349 words
ViBidLQA_v1_train                  : 367,480 words

Sampling quangbut files (takes ~1 min)...
  [WARN] CSV sampling failed for sent_truncated_zac_train.csv: 'utf-8' codec can't decode byte 0xff in position 0: invalid start byte
  sent_truncated_zac_train.csv                       (27 MB) -> est. 0 words
  [WARN] CSV sampling failed for sent_truncated_dvc_train.csv: 'utf-8' codec can't decode byte 0xff in position 0: invalid start byte
  sent_truncated_dvc_train.csv                       (42 MB) -> est. 0 words
  [WARN] CSV sampling failed for vbpl.csv: 'utf-8' codec can't decode byte 0xff in position 0: invalid start byte
  vbpl.csv                                           (1487 MB) -> est. 0 words
  [WARN] CSV sampling failed for sent_truncated_vbpl_update.csv: 'utf-8' codec c

In [40]:
# ============================================================
# Cell 9 — Save acquisition report to LOG_DIR
# ============================================================
import json, os
from datetime import datetime, timezone

report = {
    "stage":            "01_data_acquisition",
    "timestamp":        datetime.now(timezone.utc).isoformat(),
    "kaggle_datasets":  {k: {"found": True, "path": v} for k, v in KAGGLE_REGISTRY.items()},
    "hf_downloads":     hf_download_log,
    "zalo_article_count": len(all_articles),
    "token_estimates":  token_stats,
    "total_bpe_tokens": grand_total,
    "cpt_ready":        grand_total >= 200_000_000,
    "output_paths": {
        "raw_dir":  RAW_DIR,
        "hf_dir":   HF_OUT,
        "zalo_dir": ZALO_OUT,
        "log_dir":  LOG_DIR,
    }
}

report_path = f"{LOG_DIR}/stage01_acquisition_report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(f"Report saved: {report_path}")
print()
print("=" * 55)
print("STAGE 1 SUMMARY")
print("=" * 55)
print(f"  Kaggle datasets   : {len(KAGGLE_REGISTRY)}")
print(f"  HF datasets       : {sum(1 for v in hf_download_log.values() if v > 0)}")
print(f"  Zalo articles     : {len(all_articles):,}")
print(f"  Total BPE tokens  : {grand_total:,}")
print(f"  CPT ready         : {'YES' if grand_total >= 200_000_000 else 'NO'}")
print()
print("Next: notebooks/02_data_preprocessing.ipynb")

Report saved: /kaggle/working/logs/stage01_acquisition_report.json

STAGE 1 SUMMARY
  Kaggle datasets   : 3
  HF datasets       : 4
  Zalo articles     : 61,425
  Total BPE tokens  : 217,833,216
  CPT ready         : YES

Next: notebooks/02_data_preprocessing.ipynb
